# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library. All references to dataset entities (record sets, fields/columns) are made using their `@id` values.

### Dataset Source
- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- **Citation**: Kulka, M., Rodriguez Miera, S., Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.
- **Description**: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the Croissant-based dataset using `mlcroissant`. We'll load the metadata object, which will let us explore record sets, fields, and more via their `@id` values.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

List the available record sets and their fields/columns, referencing each by their `@id`.

To browse, we'll examine the `record_sets` property and drill down with their `@id`s. 

In [ ]:
# List all record sets and their fields using their @id
print("Available Record Sets:\n---------------------")
record_set_objs = metadata.record_sets
if not record_set_objs:
    print("No record sets found in the metadata.")
else:
    for recset in record_set_objs:
        print(f"- Record set name: {recset.name}")
        print(f"  @id: {recset.id}")
        if recset.fields:
            print("  Fields/Columns:")
            for field in recset.fields:
                print(f"    · {field.name} (@id: {field.id}, data_type: {field.data_type})")
        else:
            print("  No fields found.")
        print()
    # Store just @ids for later extraction
    record_set_ids = [recset.id for recset in record_set_objs]

## 3. Data Extraction

Now, we will load data from each record set using the `@id` values found above and convert each to a pandas DataFrame. This allows easy inspection and analysis.

In [ ]:
dataframes = {}
# We'll try loading all available record sets
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
        print("-----\n")
    except Exception as e:
        print(f"Failed to load record set '@id': {record_set_id}. Error: {e}")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate EDA on the main protein analysis table. Please modify the IDs below based on the actual output of the Data Overview step. All field and record set references are made using the corresponding `@id`.

*Example: Filtering by molecular weight (`cr:MolecularWeight`), normalizing, and grouping by a descriptive protein field, using their `@id`.*

In [ ]:
# You may need to update IDs below based on Data Overview output:
# Example @id values for the main protein table and columns:
PROTEIN_RECORD_SET_ID = record_set_ids[0] if record_set_ids else None  # e.g., 'cr:ProteinTable'
# Replace below with the correct @ids if available from cell 5 or metadata dump.
NUMERIC_FIELD_ID = None
GROUP_FIELD_ID = None

if PROTEIN_RECORD_SET_ID and dataframes.get(PROTEIN_RECORD_SET_ID) is not None:
    df = dataframes[PROTEIN_RECORD_SET_ID]
    print(f"Columns in DataFrame: {df.columns.tolist()}")

    # Try to guess a numeric field by type or name
    if NUMERIC_FIELD_ID is None:
        for col in df.columns:
            if df[col].dtype in [float, int] or 'weight' in col.lower() or 'coverage' in col.lower() or 'peptide' in col.lower():
                NUMERIC_FIELD_ID = col
                break
    # Try to guess a grouping field
    if GROUP_FIELD_ID is None:
        for col in df.columns:
            if col != NUMERIC_FIELD_ID and (df[col].dtype == object or 'desc' in col.lower() or 'protein' in col.lower()):
                GROUP_FIELD_ID = col
                break

    print(f"Using numeric field '@id': {NUMERIC_FIELD_ID}")
    print(f"Using group field '@id': {GROUP_FIELD_ID}")

    # Remove outliers and filter by threshold (change threshold as needed)
    if NUMERIC_FIELD_ID:
        if pd.api.types.is_numeric_dtype(df[NUMERIC_FIELD_ID]):
            threshold = df[NUMERIC_FIELD_ID].mean()  # example: mean as threshold
            filtered_df = df[df[NUMERIC_FIELD_ID] > threshold].copy()
            print(f"Filtered records with {NUMERIC_FIELD_ID} > {threshold:.2f} (n={len(filtered_df)})")
            print(filtered_df.head())

            norm_col = f"{NUMERIC_FIELD_ID}_normalized"
            filtered_df[norm_col] = (filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()) / filtered_df[NUMERIC_FIELD_ID].std()
            print(f"\nNormalized {NUMERIC_FIELD_ID} for filtered records (sample):")
            print(filtered_df[[NUMERIC_FIELD_ID, norm_col]].head())

            # Grouping
            if GROUP_FIELD_ID and GROUP_FIELD_ID in filtered_df.columns:
                grouped_df = filtered_df.groupby(GROUP_FIELD_ID).mean(numeric_only=True)
                print(f"\nGrouped by {GROUP_FIELD_ID} (mean of numeric fields):")
                print(grouped_df.head())
        else:
            print(f"Column '{NUMERIC_FIELD_ID}' is not numeric; cannot filter or normalize.")
    else:
        print(f"No numeric field found in {PROTEIN_RECORD_SET_ID}.")
else:
    print("Could not find the main record set/dataframe for EDA.")

## 5. Visualization

Let's visualize the distribution of one of the main numeric fields (e.g., protein molecular weight or similar abundance measures) and the summary by protein description (or other group field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if PROTEIN_RECORD_SET_ID and NUMERIC_FIELD_ID and PROTEIN_RECORD_SET_ID in dataframes:
    df = dataframes[PROTEIN_RECORD_SET_ID]
    if pd.api.types.is_numeric_dtype(df[NUMERIC_FIELD_ID]):
        plt.figure(figsize=(8,4))
        sns.histplot(df[NUMERIC_FIELD_ID].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {NUMERIC_FIELD_ID}")
        plt.xlabel(NUMERIC_FIELD_ID)
        plt.show()

        # If group field is available, show top categories by mean numeric field
        if GROUP_FIELD_ID in df.columns:
            plt.figure(figsize=(10,4))
            grouped = df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().sort_values(ascending=False).head(15)
            sns.barplot(x=grouped.values, y=grouped.index)
            plt.title(f"Top 15 {GROUP_FIELD_ID} by mean {NUMERIC_FIELD_ID}")
            plt.xlabel(f"Mean {NUMERIC_FIELD_ID}")
            plt.ylabel(GROUP_FIELD_ID)
            plt.tight_layout()
            plt.show()
    else:
        print(f"Field {NUMERIC_FIELD_ID} is not numeric, skipping visualization.")
else:
    print("Cannot plot: missing data, numeric field or record set.")

## 6. Conclusion

In this notebook, we:
- Loaded Croissant metadata and record sets from the FAIR^2 protein analysis dataset.
- Explored available record sets and fields using `@id` references.
- Extracted a main protein records table and demonstrated basic EDA: filtering, normalization, and group aggregation.
- Visualized distributions of a numeric field and group aggregates.

**Next steps:** You can adapt this notebook by specifying the exact `@id` values for numeric and grouping fields depending on your research interest or by further joining across record sets.

For new Croissant datasets, simply change the schema URL, and re-run the notebook to explore the structure and extract data using each entity's `@id`.